# Speech Denoising — Model Architecture & Training

Baseline CNN model for speech denoising on VoiceBank+DEMAND dataset.

**Architecture:** 1D CNN (2 layers) | **Loss:** MSELoss | **Optimizer:** Adam

---

## 1. Imports

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import librosa as lb
import matplotlib.pyplot as plt
import soundfile as sf
from pathlib import Path
from IPython.display import Audio
from tqdm import tqdm

## 2. Load Preprocessed Data

Load train chunks saved from notebook 01.

In [ ]:
train_clean_chunks = np.load('data/train_clean_chunks.npy')
train_noisy_chunks = np.load('data/train_noisy_chunks.npy')

In [ ]:
train_clean_chunks.shape

## 3. Model Architecture

**DenoisingModel** — 2-layer 1D CNN.

- Input: `(batch, 16000)` — noisy waveform
- Conv1d(1→16, kernel=3, padding=1) + LeakyReLU — extract local patterns
- Conv1d(16→1, kernel=3, padding=1) — reconstruct clean waveform
- Output: `(batch, 16000)` — predicted clean waveform

unsqueeze/squeeze handle the channel dimension required by Conv1d.

In [ ]:
class DenoisingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu = nn.LeakyReLU()
        self.layer1 = nn.Conv1d(1, 16, 3, padding=1)
        self.layer2 = nn.Conv1d(16, 1, 3, padding=1)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = x.squeeze(1)
        return x

In [ ]:
model = DenoisingModel()
print(model) 

## 4. Loss Function & Optimizer

**MSELoss** — Mean Squared Error between predicted and clean waveform.
**Adam** — adaptive learning rate optimizer, lr=0.001.

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001 )

## 5. Dataset & DataLoader

Convert numpy arrays to PyTorch tensors and create batched DataLoader (batch_size=32).

In [ ]:
torch_train_clean_chunks = torch.from_numpy(train_clean_chunks)
torch_train_noisy_chunks = torch.from_numpy(train_noisy_chunks)

In [ ]:
torch_train_clean_chunks.shape


In [ ]:
torch_train_noisy_chunks.shape

In [ ]:
train_dataset = torch.utils.data.TensorDataset(torch_train_noisy_chunks, torch_train_clean_chunks)

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=32,)

In [ ]:
len(train_dataloader)

## 6. Training Loop

3 epochs on CPU. Loss is averaged across all batches per epoch.

In [ ]:
for epoch in tqdm(range(3)):
    epoch_loss = 0
    for batch in train_dataloader:
        noisy_batch, clean_batch = batch
        optimizer.zero_grad()
        train_pred_clean = model(noisy_batch)
        train_loss = loss_fn(train_pred_clean, clean_batch)
        train_loss.backward()
        optimizer.step()
        epoch_loss += train_loss.item()
    avg_epoch_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch {epoch}, Avg Loss: {avg_epoch_loss:.4f}")


In [ ]:
train_loss.item()

## 7. Save Model Weights

In [ ]:
torch.save(model.state_dict(), 'denoising_cnn_2layers_3epochs.pth',)

## 8. Inference

Run a single test file through the trained model and compare noisy vs predicted clean audio.

In [ ]:
test_noisy, _ = lb.load('data/archive/noisy_testset_wav/p232_011.wav', sr=None)

In [ ]:
test_noisy_tensor = torch.from_numpy(test_noisy).unsqueeze(0)

In [ ]:
# verify model output shape
print(model(test_noisy_tensor).shape)


In [ ]:
pred_clean = model(test_noisy_tensor).detach().numpy().squeeze(0)


In [ ]:
# verify audio output clean
Audio(pred_clean, rate=16000)

In [ ]:
# compare to noisy audio output
Audio(test_noisy, rate=16000)